In [1]:
# conda env 'at3d_pure_k9'
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent

import at3d
from collections import OrderedDict
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
fm_file = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V05' / 'default' / 'iterations' / 'iter006.nc'

save_LUT = True
LUT_outfile = PROJECT_ROOT / 'NK1990_1D_Retrieval' / 'LUT_I_V05_periodic.nc'

sensors, solvers, rte_grid = at3d.util.load_forward_model(fm_file)

wavelength = sensors.get_unique_solvers()
assert len(wavelength == 1)
wavelength = wavelength[0]

## Create rte_grid with same vertical grid, but only a 3 x 3 in the horizontal
rte_grid = at3d.grid.make_grid(rte_grid.delx.data, 3, rte_grid.dely.data, 3, rte_grid.z.data)

## Define cloud base and cloud top
cb = 0.4
ct = 0.7
i_low = np.argwhere(rte_grid.z.data >= cb)[0][0]
i_up = np.argwhere(rte_grid.z.data <= ct)[-1][0]
thickness = rte_grid.z.data[i_up] - rte_grid.z.data[i_low]
## Define optical depths in LUT
taus = np.logspace(-1, 1.9, 100)
taus[0] = 0

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:208: UserWarning: Splitting accuracy parameter is not a fraction but scales with fluxes. splitacc/nominal flux = 2.2092107825441377e-05
  warnings.warn("Splitting accuracy parameter is not a fraction but scales with fluxes."
/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '2'. Max cell optical depth: '2.1719627380371094'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


In [3]:
## Make a sensor to observe radiance
LUT_sensors = at3d.containers.SensorsDict()
LUT_sensor = at3d.sensor.orthographic_projection(
    wavelength=wavelength,
    bounding_box=rte_grid,
    x_resolution=rte_grid.delx.data,
    y_resolution=rte_grid.dely.data,
    azimuth=0,
    zenith=0,
    altitude='TOA',
    stokes='I',
    sub_pixel_ray_args={'method': None}
)
LUT_sensors.add_sensor('LUT_sensor', LUT_sensor)

In [4]:
## Set up LUT solver with the features of the retreival solver

# Numerical parameters
numerical_parameters = solvers[wavelength].numerical_params
numerical_parameters['ip_flag'] = 3 # Set to independent pixel approximation
numerical_parameters['y_boundary_condition'] = 'periodic'
numerical_parameters['x_boundary_condition'] = 'periodic'

# Surface
surface = solvers[wavelength].surface

# Source
source = solvers[wavelength].source

# Medium
medium = OrderedDict()
medium['background'] = solvers[wavelength].medium['background'].isel(x=slice(0,3), y=slice(0,3))
medium['cloud'] = solvers[wavelength].medium['cloud'].isel(x=slice(0,3), y=slice(0,3))

In [5]:
Is = np.zeros(taus.shape)
num_unique = np.zeros(taus.shape)
diff = np.zeros(taus.shape)
LUT_solvers = at3d.containers.SolversDict()
for i, tau in enumerate(taus):
    ext = np.zeros((rte_grid.dims['x'], rte_grid.dims['y'], rte_grid.dims['z']))
    ext[:, :, i_low:i_up] = tau/thickness
    medium['cloud']['extinction'] = (['x','y','z'], ext)

    LUT_solver = at3d.solver.RTE(
        numerical_params = numerical_parameters,
        medium = medium,
        source = source,
        surface = surface,
        num_stokes = 1
    )
    LUT_solvers.add_solver(wavelength, LUT_solver)
    LUT_sensors.get_measurements(LUT_solvers)

    cam = LUT_sensors['LUT_sensor']['sensor_list'][0]
    num_unique[i] = len(np.unique(cam.I.data))
    diff[i] = np.max(cam.I.data) - np.min(cam.I.data)
    # assert num_unique[i] <= 3, "Tau = {}\n{}".format(tau, cam.I)
    Is[i] = np.mean(cam.I.data)

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:208: UserWarning: Splitting accuracy parameter is not a fraction but scales with fluxes. splitacc/nominal flux = 2.2092107825441377e-05
  warnings.warn("Splitting accuracy parameter is not a fraction but scales with fluxes."


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -1.013  0.522E+02      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.462  0.973E+00      198   256.00  1.000   [Radiance 0.670 micron]
     3  -1.866  0.885E+00      216   256.00  1.000   [Radiance 0.670 micron]
     4  -2.213  0.904E+00      216   256.00  1.000   [Radiance 0.670 micron]
     5  -2.810  0.904E+00      216   256.00  1.000   [Radiance 0.670 micron]
     6  -3.074  0.816E-01      540   256.00  1.000   [Radiance 0.670 micron]
     7  -3.512  0.569E-01      576   256.00  1.000   [Radiance 0.670 micron]
     8  -4.528  0.569E-01      576   256.00  1.000   [Radiance 0.670 micron]
 ! Iterations:      8     Final Criterion:  0.000030   [Radiance 0.670 micron]
  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -1.019  0.522E+02      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.466  0.974E+00      252   256.00  1.000   [Radiance 0.670 micron]
   

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:208: UserWarning: Splitting accuracy parameter is not a fraction but scales with fluxes. splitacc/nominal flux = 2.2092107825441377e-05
  warnings.warn("Splitting accuracy parameter is not a fraction but scales with fluxes."


     7  -3.490  0.222E+00      577   256.00  1.000   [Radiance 0.670 micron]
     8  -4.311  0.221E+00      577   256.00  1.000   [Radiance 0.670 micron]
 ! Iterations:      8     Final Criterion:  0.000049   [Radiance 0.670 micron]
  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -1.018  0.522E+02      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.466  0.974E+00      252   256.00  1.000   [Radiance 0.670 micron]
     3  -1.861  0.887E+00      306   256.00  1.000   [Radiance 0.670 micron]
     4  -2.073  0.907E+00      306   256.00  1.000   [Radiance 0.670 micron]
     5  -2.696  0.907E+00      306   256.00  1.000   [Radiance 0.670 micron]
     6  -2.926  0.221E+00      576   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1010  Npnts=     577


/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:208: UserWarning: Splitting accuracy parameter is not a fraction but scales with fluxes. splitacc/nominal flux = 2.2092107825441377e-05
  warnings.warn("Splitting accuracy parameter is not a fraction but scales with fluxes."


...

  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.939  0.796E+03      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.710  0.791E+03      135   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1010  Npnts=     577
     3  -1.482  0.241E+02      577   256.00  1.000   [Radiance 0.670 micron]
     4  -1.507  0.235E+02      577   256.00  1.000   [Radiance 0.670 micron]
     5  -1.757  0.231E+02      577   256.00  1.000   [Radiance 0.670 micron]
     6  -1.899  0.230E+02      577   256.00  1.000   [Radiance 0.670 micron]
     7  -2.049  0.229E+02      577   256.00  1.000   [Radiance 0.670 micron]
     8  -2.161  0.229E+02      577   256.00  1.000   [Radiance 0.670 micron]
     9  -2.357  0.237E+02      577   256.00  1.000   [Radiance 0.670 micron]
    10  -2.492  0.240E+02      577   256.00  1.000   [Radiance 0.670 micron]
    11  -2.608  0.247E+02      577   256.00  1.000   [Radiance 0.

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:208: UserWarning: Splitting accuracy parameter is not a fraction but scales with fluxes. splitacc/nominal flux = 2.2092107825441377e-05
  warnings.warn("Splitting accuracy parameter is not a fraction but scales with fluxes."
/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '16'. Max cell optical depth: '30.59096908569336'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.939  0.799E+03      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.711  0.794E+03      135   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1010  Npnts=     577
     3  -1.479  0.258E+02      577   256.00  1.000   [Radiance 0.670 micron]
     4  -1.505  0.252E+02      577   256.00  1.000   [Radiance 0.670 micron]
     5  -1.755  0.247E+02      577   256.00  1.000   [Radiance 0.670 micron]
     6  -1.897  0.246E+02      577   256.00  1.000   [Radiance 0.670 micron]
     7  -2.045  0.244E+02      577   256.00  1.000   [Radiance 0.670 micron]
     8  -2.152  0.245E+02      577   256.00  1.000   [Radiance 0.670 micron]
     9  -2.332  0.248E+02      577   256.00  1.000   [Radiance 0.670 micron]
    10  -2.469  0.248E+02      577   256.00  1.000   [Radiance 0.670 micron]
    11  -2.592  0.249E+02      577   256.00  1.000   [Radiance 0.

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:208: UserWarning: Splitting accuracy parameter is not a fraction but scales with fluxes. splitacc/nominal flux = 2.2092107825441377e-05
  warnings.warn("Splitting accuracy parameter is not a fraction but scales with fluxes."
/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '16'. Max cell optical depth: '32.72433090209961'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.939  0.801E+03      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.711  0.796E+03      135   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1010  Npnts=     577
     3  -1.475  0.276E+02      577   256.00  1.000   [Radiance 0.670 micron]
     4  -1.502  0.269E+02      577   256.00  1.000   [Radiance 0.670 micron]
     5  -1.753  0.264E+02      577   256.00  1.000   [Radiance 0.670 micron]
     6  -1.896  0.263E+02      577   256.00  1.000   [Radiance 0.670 micron]
     7  -2.041  0.261E+02      577   256.00  1.000   [Radiance 0.670 micron]
     8  -2.144  0.262E+02      577   256.00  1.000   [Radiance 0.670 micron]
     9  -2.306  0.265E+02      577   256.00  1.000   [Radiance 0.670 micron]
    10  -2.444  0.265E+02      577   256.00  1.000   [Radiance 0.670 micron]
    11  -2.572  0.267E+02      577   256.00  1.000   [Radiance 0.

In [6]:
LUT = xr.Dataset(
    coords = dict(I=Is),
    data_vars = dict(
        tau=(['I'], taus),
        num_unique = (['I'], num_unique),
        diff = (['I'], diff)
    )
)
assert np.all(np.diff(LUT.tau.data) > 0), "LUT tau not monotonic"
assert np.all(np.diff(LUT.I.data) > 0), "LUT I not monotonic"
if(save_LUT):
    LUT.to_netcdf(LUT_outfile)

In [7]:
# f, ax = plt.subplots(1, 1, figsize=(7,5))
# LUT.tau.plot(y='I', ax=ax)
# s = ax.scatter(LUT.tau, LUT.I, c=LUT.num_unique)
# ax.set_xscale('log')
# plt.colorbar(s, label='Num Unique')

In [8]:
# f, ax = plt.subplots(1, 1, figsize=(7,5))
# LUT.tau.plot(y='I', ax=ax)
# s = ax.scatter(LUT.tau, LUT.I, c=LUT.num_unique)
# # ax.set_xscale('log')
# plt.colorbar(s, label='Num Unique')

In [9]:
# f, ax = plt.subplots(1, 1, figsize=(7,5))
# LUT.tau.plot(y='I', ax=ax)
# s = ax.scatter(LUT.tau, LUT.I, c=LUT['diff'])
# ax.set_xscale('log')
# plt.colorbar(s, label='Difference')

In [10]:
# f, ax = plt.subplots(1, 1, figsize=(7,5))
# # LUT.tau.plot(y='I', ax=ax)
# s = ax.scatter(LUT.I, LUT['diff'])
# ax.set_yscale('log')
# ax.set_xlabel('I')
# ax.set_ylabel('Difference')